In [4]:
import os
import certifi
import requests

from dotenv import load_dotenv

from langchain_community.tools.tavily_search import TavilySearchResults
from langchain.tools import tool

# NEW (instead of from langchain import hub)
from langchain_classic import hub

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_11968\80028935.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools.tavily_search import TavilySearchResults


In [4]:
from langchain.agents import create_agent

c:\Users\Lenovo\OneDrive\Desktop\Agentic_Ai_project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
# ==========================================
# LOAD ENV VARIABLES
# ==========================================

os.environ["SSL_CERT_FILE"] = certifi.where()

load_dotenv()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")


In [6]:
search_tool = TavilySearchResults(max_results=2)

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_11968\919418145.py:1: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  search_tool = TavilySearchResults(max_results=2)


In [7]:
result = search_tool.invoke("Give me the latest news on AI")
result

[{'title': 'Google AI announcements from May 2026',
  'url': 'https://blog.google/innovation-and-ai/technology/ai/google-ai-updates-may-2026',
  'content': 'The big picture\n\nMay 2026 was packed with AI news. At Google I/O 2026, we officially entered the agentic Gemini era with the launch of Gemini 3.5 — which delivers frontier intelligence for agents and coding — and Gemini Omni, where Gemini’s ability to reason meets the ability to create. The Android Show set the stage with brand-new hardware built specifically for these tools, including the Googlebook from our hardware partners. We also broadened our personal wellness tools with the new Google Health app and Fitbit Air, and launched an initiative to apply advanced quantum science and AI to the life sciences. Ultimately, May was all about making AI more proactive, helpful and integrated into your everyday life.\n\nAI that helps you create and explore [...] Learn more:\n\nLearn more:\n\nLearn more:\n\nLearn more:\n\nLearn more:\n\nL

In [7]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

In [9]:
response = llm.invoke("Tell me a joke about AI")
response

AIMessage(content="Why did the AI break up with the calculator?\n\nBecause it said they had too many *issues* and it couldn't *compute* their future together!", additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f12dd-0198-78a0-8dee-566474252fd1-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 7, 'output_tokens': 1341, 'total_tokens': 1348, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 1308}})

In [8]:
# ==========================================
# PROMPT
# ==========================================

prompt = hub.pull("hwchase17/react")

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_11968\2052123876.py:5: LangChainDeprecationWarning: langchain_classic.hub.pull is deprecated. Use the LangSmith SDK instead.
  prompt = hub.pull("hwchase17/react")


In [9]:
prompt

PromptTemplate(input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'], input_types={}, partial_variables={}, metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'}, template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}')

In [14]:
print(type(prompt))
print(prompt)

<class 'langchain_core.prompts.prompt.PromptTemplate'>
input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'] input_types={} partial_variables={} metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'} template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}'


In [15]:
system_prompt = """
You are a helpful AI assistant.

You have access to the following tools:
- Search Tool
- Weather Tool

Always think carefully before answering.
If required, use the appropriate tool.
Give concise and accurate answers.
"""

In [10]:
from langchain.tools import tool
import requests
import os

@tool
def get_weather_data(city: str) -> str:
    """
    Returns current weather information for a city.
    """

    api_key = os.getenv("OPENWEATHER_API_KEY")

    url = f"https://api.openweathermap.org/data/2.5/weather?q={city}&appid={api_key}&units=metric"

    response = requests.get(url)

    if response.status_code != 200:
        return "Could not fetch weather."

    data = response.json()

    weather = data["weather"][0]["description"]
    temp = data["main"]["temp"]
    humidity = data["main"]["humidity"]

    return (
        f"Weather in {city}:\n"
        f"Temperature: {temp}°C\n"
        f"Humidity: {humidity}%\n"
        f"Condition: {weather}"
    )

In [11]:
# ==========================================
# TOOLS
# ==========================================

tools = [search_tool, get_weather_data]

In [16]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=system_prompt
)

In [18]:
response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "Find the capital of India "
                    "and then find its current weather."
                )
            }
        ]
    }
)

In [20]:
print(response["messages"][-1].content)

[{'type': 'text', 'text': 'The capital of India is New Delhi. The current weather in New Delhi is 43.26°C, with 20% humidity and broken clouds.', 'extras': {'signature': 'CoICAQw51sdsToR5EJRdTyUEpE++6hAc51h16xYQbar9uuRnwj3RTJKzrO9l//X9zrrDrthcODU/ZRMrEuqkXFVE/rr+lxNFANuwytsdPpkX7y3mQNIySj6hBvRLgCEi5dcnDuhgqienJkmoc09iWQuAGS/V1P+5jmvyQPy//NunzM4d769ce002LBsT4SaE5SfUlMiJXzD4+z6nd0KnACYOf+cE4EowWG3NMQNipmccNM2BfSwWeukyH8MnCewNVKtdf4sNCH4huDvYnrdO4ptQLJHj1Y4f3JtvLxsjrDqfhRA86C80WzPZE4XT/7aLQ0tKgbI81qdMAfP8vi+kAwA50MvB'}}]


In [21]:
for msg in response["messages"]:
    print(type(msg).__name__)
    print(msg)
    print("-" * 50)

HumanMessage
content='Find the capital of India and then find its current weather.' additional_kwargs={} response_metadata={} id='eafd4091-ba8c-4f72-8da3-c9fa7a8abbcc'
--------------------------------------------------
AIMessage
content='' additional_kwargs={'function_call': {'name': 'tavily_search_results_json', 'arguments': '{"query": "capital of India"}'}, '__gemini_function_call_thought_signatures__': {'36a48c0b-2899-4680-b144-18c9331cebd1': 'CoEDAQw51sfgTIUNkK2NecxKfw64P2e+cvjekYofTM+Nyr2mCuDB1FC9/92xuAiBQSIvlsVFwOmp6xs7kzY8jqTkx3g+y4640+bGOmPfKSWsWGwTUT9dQbFESgqoIz/Nea7dJeFAf8F0y8MpyCtqmYsmKVsBwJo7gD+YNWlD5kT359kh8o61SjHndzBy0j4cmMeGZrl1s8z0WI3Iuxl0DzU6kLaqtuIP4lQlsSp/WF3WdeXsMtztlZVlm4LdZ/Lzdba7atBZkEolQ5THq5nApIDDbwYLClntv241vNNqu9vPDnJpAdT8fjpXpC08Qdj00iCAJxMhrTRO/ZJDvPOXEnE7ztVw/y2IswvSu12Lzoq2eIDhMF9osn9QvvS2b3SwsvOvBgKkmZ+g7jn9vpfFhg7qvgMhoSdP3GswUP8RQo8JySsJOXxoKUSGP/o7c9+lEOB4GJTPPIXlTPgkqjgAP67aPhzyGHsS6DFf0Ub45Sl5vOWPGp3fEJ3B6vaqARSeg4IwtA=='}} response_metadata={'finis